# Long Short-Term Memory
In this exercise, we will implement an LSTM. In the class, we have already seen the definition of the LSTM update rules at time step $t$:

$$
\begin{align}
f_t &= \sigma(W_f h_{t-1} + U_f x_t + b_f) \\
i_t &= \sigma(W_i h_{t-1} + U_i x_t + b_i) \\
o_t &= \sigma(W_o h_{t-1} + U_o x_t + b_o) \\
\tilde{c}_t &= \tanh(W_c h_{t-1} + U_c x_t + b_c) \\
c_t &= f_t * c_{t-1} + i_t * \tilde{c}_t \\
h_t &= o_t * \tanh(c_t)
\end{align}
$$

In [1]:
import torch
import torch.nn as nn

Implement this original version of the LSTM as an `LSTMCell`.

In [3]:
class LSTMCell(nn.Module):

    def __init__(self, input_dim, hidden_dim, num_chunks=4):
        super().__init__()
        # self.w_f = nn.Parameter(torch.zeros(hidden_dim, hidden_dim))
        # self.u_f = nn.Parameter(torch.zeros(hidden_dim, input_dim))  # U_f @ x_t
        # self.w_f = nn.Linear(hidden_dim, hidden_dim, bias=False)
        # self.u_f = nn.Linear(input_dim, hidden_dim, bias=False)
        # self.b_f = nn.Parameter(torch.zeros(hidden_dim))

        self.hidden_dim = hidden_dim
        self.num_chunks = num_chunks

        self.W = nn.Linear(hidden_dim, num_chunks * hidden_dim, bias=False)
        self.U = nn.Linear(input_dim, num_chunks * hidden_dim, bias=False)
        self.b = nn.Parameter(torch.zeros(num_chunks * hidden_dim))
    
    def reset_parameters(self):
        for weight in self.parameters():
            nn.init.normal_(weight, mean=0, std=1)

    def forward(self, x, prev_cell_state, prev_hidden_state):
        # f_t = torch.sigmoid(self.W_f @ prev_hidden_state + self.U_f @ x + self.b_f)
        updates = self.W(prev_hidden_state) + self.U(x) + self.b
        updates = updates.reshape(self.num_chunks, self.hidden_dim)
        f_t = torch.sigmoid(updates[0])
        i_t = torch.sigmoid(updates[1])
        o_t = torch.sigmoid(updates[2])
        new_cell_memory = torch.tanh(updates[3])
        cell_state = f_t * prev_cell_state + i_t * new_cell_memory  # element-wise multiplication
        hidden_state = o_t * torch.tanh(cell_state)
        return cell_state, hidden_state

Create a 2-layer LSTM from your LSTMCell base class and run a forward pass with a random input sequence to test that all your dimensions are correct.

In [5]:
class LSTM(nn.Module):

    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([])
        for i in range(num_layers):
            in_dim = input_dim if i == 0 else hidden_dim
            self.layers.append(LSTMCell(in_dim, hidden_dim))
    
    def reset_parameters(self):
        for layer in self.layers:
            layer.reset_parameters()

    def forward(self, x, prev_c, prev_h):
        outputs = []
        for x_i in x:
            new_cell_states = []
            new_hidden_states = []
            for i, lstm_cell in enumerate(self.layers):
                new_c, new_h = lstm_cell(x_i, prev_c[i], prev_h[i])
                x_i = new_h
                new_cell_states.append(new_c)
                new_hidden_states.append(new_h)
            outputs.append(new_h)
            prev_c = torch.stack(new_cell_states)
            prev_h = torch.stack(new_hidden_states)
        return torch.stack(outputs), (prev_c, prev_h)

input_dim = 5
hidden_dim = 10
sequence_length = 6
num_layers = 2

lstm = LSTM(input_dim, hidden_dim, num_layers)
lstm.reset_parameters()

x = torch.randn(sequence_length, input_dim)
c0 = torch.zeros(num_layers, hidden_dim)
h0 = torch.zeros(num_layers, hidden_dim)

outputs, (cn, hn) = lstm(x, c0, h0)
print(outputs.shape)
print(hn.shape)


torch.Size([6, 10])
torch.Size([2, 10])


Implement a subclass of your LSTM that uses a coupled forget and input gate, i.e. the cell state update becomes:

$$c_t = f_t * c_{t-1} + (1-f_t) * \tilde{c}_t$$

**Bonus:** Implement *peephole connections* as described at the start of the Section *Variants on Long Short Term Memory* in [this blog post explaining LSTMs](https://colah.github.io/posts/2015-08-Understanding-LSTMs/).

The gate update definitions get an additional term that looks at the cell state:
$$
\begin{align}
f_t &= \sigma(W_f h_{t-1} + U_f x_t + b_f \boldsymbol{+ V_f c_{t-1}}) \\
i_t &= \sigma(W_i h_{t-1} + U_i x_t + b_i \boldsymbol{+ V_i c_{t-1}}) \\
o_t &= \sigma(W_o h_{t-1} + U_o x_t + b_o \boldsymbol{+ V_o c_t})
\end{align}
$$

To make the task a bit easier, we will implement the last equation with the cell state of the previous time step $t-1$ as $$o_t = \sigma(W_o h_{t-1} + U_o x_t + b_o \boldsymbol{+ V_o c_{t-1}})$$ instead.